# Reproducing DeWeese and Zador (2004) from CRCNS data set ac-2

**Paper.** DeWeese MR, Zador AM (2004). Shared and private variability in the auditory cortex.
*J Neurophysiol* 92:1840-1855. [doi:10.1152/jn.00197.2004](https://doi.org/10.1152/jn.00197.2004).
The claims file (`claims/crcns-ac-2/10.1152-jn.00197.2004.yaml`) was written from the version of
record (text extracted from the journal PDF, with its Methods and figure legends; the paper has no
supplement and links no code).

**Data set.** [CRCNS ac-2](https://crcns.org/data-sets/ac/ac-2), doi:10.6080/K0G44N6R, read from
the public bucket `crcnsarchive` (`ac-2/crcns-ac-2.tar.gz` and
`ac-2/crcns-ac-2-docs_and_scripts.tar.gz`). CRCNS data sets are not versioned, so the version is
recorded as `unversioned`; the only published content is this one, and the paper cites none. It
is pinned by the MD5 sums of the data set's `checksums.md5`: `crcns-ac-2.tar.gz`
af006ca3530e7f615a37d0548ba86cf2 and `crcns-ac-2-docs_and_scripts.tar.gz`
6ffd46575e78b3f8a2ad3efb9f95008a. The notebook computes both sums from the bytes it streams and
compares them (below), so a later change of the files is detected. The data tar is a gzipped tar,
which cannot be read by range; it is streamed once into memory (about 30 s) and nothing is
written to disk.

**What the data set holds.** The 33 anesthetized whole-cell recordings of the variability
analysis, 29 of them with a simultaneous LFP; six recordings after TTX (the paper's mEPSPs); one
awake recording that belongs to a later paper and is not used.

| Claim | Tested | Outcome |
|---|---|---|
| fig2-stereotyped-shape | yes | partially reproduced |
| fig3a-reliable-neuron | yes | reproduced |
| fig3b-total-variability-distribution | yes | reproduced (assumed) |
| total-variability-population | yes | reproduced (assumed) |
| lfp-culling-reduction | yes | reproduced (assumed) |
| fig5b-residual-lfp-psp-correlation | yes | reproduced |
| fig7a-lfp-culling-estimation | yes | partially reproduced (assumed) |
| psp-culling-reduction | yes | partially reproduced |
| fig4c-psp-culling-can-increase | yes | partially reproduced (assumed) |
| fig7b-private-best-estimate | yes | reproduced |
| culling-controls | yes | not reproduced |
| fig7d-large-responses | yes | reproduced |
| large-responses-controls | yes | not reproduced |
| cv-large-responses | yes | partially reproduced |
| lfp-cv-not-different | yes | reproduced |
| anesthesia-comparison | yes | partially reproduced |
| fig8-spontaneous-veto | yes | reproduced |
| overfitting-correction-example | yes | reproduced |
| private-shared-ranges | yes | partially reproduced |
| quantal-comparison | yes | reproduced |
| mepsp-size | yes, every reading run | blocked: methods underspecified |
| recording-properties | yes | reproduced (assumed) |
| access-resistance-depth-no-effect | yes | reproduced (assumed) |

"Assumed" marks outcomes that rest on a choice the paper does not make and that would differ
under another reading (the spike rule for most; the input resistance for recording-properties).

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ac2

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
D = ac2.load()
N = D["neurons"]
print(f"data set {ac2.DATASET} ({ac2.DATASET_VERSION}, doi:{ac2.DATASET_DOI}) loaded in {time.time() - started:.0f} s")
print("MD5 of the streamed files:", D["md5"])
print("MD5 in checksums.md5:     ", ac2.MD5)
print("Same:", D["md5"] == ac2.MD5)

## Data check

Each check of the documentation against the values is stated under its output.

In [ ]:
info = pd.DataFrame(D["info"]).T
print("Records in neuron_recording_info.mat:", len(info))
print("Anesthetized neurons with traces:", len(N), "; TTX recordings:", len(D["ttx"]),
      "; files of the awake recording (not used):", len(D["awake_files"]))
print("Other files in the data tar:", D["other_files"])
print("Tone order read from Anesthetized_cell_info.txt (Hz):", ac2.TONE_ORDER.tolist())
steps = np.diff(np.log(np.sort(ac2.TONE_ORDER)))
print(f"{len(ac2.TONE_ORDER)} tones, {ac2.TONE_ORDER.min()} to {ac2.TONE_ORDER.max()} Hz; log steps {steps.min():.4f} to {steps.max():.4f}")
rows = []
for name, n in N.items():
    st = ac2.stable(n)
    p = n["params"]
    rows.append({
        "neuron": name, "anesthetic": n["info"]["which_anesthetic"], "LFP": int(n["info"]["was_LFP_recorded"]),
        "lfp_gain": n["info"]["axogaincorrection_LFP"], "sweeps": n["n_sweeps"], "traces": len(n["r"]),
        "cycles": int(len(n["r"]) // 32), "first_cycle": int(p["first_cycle"]), "last_cycle": int(p["last_cycle"]),
        "reps": int(p["last_cycle"] - p["first_cycle"] + 1), "stable_traces": int(st.sum()),
        "spikethresh": p["spikethresh"], "spikes_cell": int((ac2.spikes(n, "cell") & st).sum()),
        "spikes_-20mV": int((ac2.spikes(n, "abs") & st).sum()),
        "R_access": n["info"]["R_access"], "depth_um": n["info"]["Sutter_diag_depth"] / np.sqrt(2),
        "max_baseline_offset": float(np.abs(n["r"][:, :ac2.PRE].mean(1)).max()),
        "trace_vs_sweep_maxdiff": float(n["trace_match_error"].max()),
        "onset_shift_samples": int(np.abs(n["onset_offset"]).max()),
    })
cells = pd.DataFrame(rows).set_index("neuron")
print(cells.round(3).to_string())
print("\nNeurons by anesthetic:", cells.anesthetic.value_counts().to_dict(), "(paper: 13 P, 3 P with diazepam, 17 urethane)")
print("Neurons with the LFP flag:", int(cells.LFP.sum()), "(paper: 29 with a stable LFP)")
print("Traces = 16 x sweeps in every neuron:", bool((cells.traces == 16 * cells.sweeps).all()))
print("Largest |mean of the first 15 ms| of any trace (should be 0):", cells.max_baseline_offset.max())
print("Largest difference between a trace and its place in the sweep (mV):", cells.trace_vs_sweep_maxdiff.max(),
      "; largest onset shift found (samples):", cells.onset_shift_samples.max())

The data set's files agree with its documentation on the counts: 33 anesthetized neurons, 13
under pentobarbital, 3 with diazepam and 17 under urethane (the paper's group sizes), and 29 with
the flag `was_LFP_recorded` (the paper's 29 with a stable LFP). Each trace is 600 samples (150 ms
at the documented 4 kHz) and starts at its nominal place in the sweeps (200 ms + 500 ms per tone,
minus 15 ms): every trace equals its piece of the sweep minus its stored baseline, to floating
point, and no shift is needed. The documentation says the stimulus-locked traces were corrected
for a mismatch of the tone onset times; the traces are not shifted against the stored sweeps, so
either the sweeps carry the same correction or none was applied; this cannot be told from the
files. Every trace is relative to the mean of its first 15 ms, as documented. Each neuron's
`cell_specific_parameters.m` gives a stable epoch (`first_cycle` to `last_cycle`, cycles of 32
tones) and a spike threshold; the files do not describe them for these neurons (the TTX
neurons' copies of the file describe both fields).

In [ ]:
reps = cells.reps
print("Repetitions per tone in the stable epoch: mean %.1f, SE %.1f, range %d to %d, n = %d"
      % (reps.mean(), reps.std(ddof=1) / np.sqrt(len(reps)), reps.min(), reps.max(), len(reps)))
print("Paper: 18 +- 2 (mean +- SE), range 10 to 63, n = 33")
# Recordings deposited twice: compare the first sweep of every pair of neurons.
first = {name: n["r"][:64].astype(float) for name, n in N.items()}
dup = [(a, b) for i, a in enumerate(first) for b in list(first)[i + 1:]
       if first[a].shape == first[b].shape and np.allclose(first[a], first[b])]
print("Pairs of neurons with identical first two sweeps:", dup)
print("Animals by recording date prefix:", len({c[:6] for c in N}), "(paper: 21 animals)")
print("Ages and dates are in the docs table only; the date prefix of each name is the recording day.")

Repetitions per tone are taken as the length of each neuron's stable epoch. They match the paper:
mean 18, range 10 to 63 (printed above). No two neurons share their traces. The neuron names
carry 21 recording days, the paper's 21 animals; names with the same day are listed as the same
animal in the docs table ("same").

In [ ]:
# The clock of the traces against the tone sequence: the tone explains the response after onset
# (peak in the 50 ms after onset), and not the 100 ms before onset, taken from the sweeps. The
# null is the same measure with the tone labels permuted at random over the trials.
def eta2(h, labels):
    df = pd.DataFrame({"h": h, "g": labels})
    ss_b = (df.groupby("g").h.transform("mean") - df.h.mean()) ** 2
    return ss_b.sum() / ((df.h - df.h.mean()) ** 2).sum()

rng_clock = np.random.default_rng(1)
clock = []
for name, n in N.items():
    idx = np.flatnonzero(ac2.stable(n) & ~ac2.spikes(n, "cell"))
    labels = n["tone"][idx]
    post = n["r"][idx, ac2.PRE:ac2.PRE + 200].max(1)
    pre = n["pre_wc"][idx].max(1) - n["pre_wc"][idx].mean(1)
    null = [eta2(post, rng_clock.permutation(labels)) for _ in range(20)]
    row = {"neuron": name, "post_onset": eta2(post, labels), "pre_onset": eta2(pre, labels),
           "null_median": np.median(null), "null_max": np.max(null)}
    if ac2.has_lfp(n):
        lpost = np.abs(n["lfp"][idx, ac2.PRE:ac2.PRE + 200]).max(1)
        lpre = np.abs(n["pre_lfp"][idx] - n["pre_lfp"][idx].mean(1, keepdims=True)).max(1)
        row.update(lfp_post=eta2(lpost, labels), lfp_pre=eta2(lpre, labels),
                   lfp_null_max=max(eta2(lpost, rng_clock.permutation(labels)) for _ in range(20)))
    row["evoked_mV"] = float(n["r"][idx, ac2.PRE + 40:ac2.PRE + 200].mean())
    # Response onset: first time after onset where the mean over all tones exceeds 1 mV, and the
    # largest |mean| in the 15 ms before onset and the first 5 ms after it.
    grand = n["r"][idx].mean(0)
    above = np.flatnonzero(grand[ac2.PRE:] > 1.0)
    row["onset_latency_ms"] = above[0] / ac2.FS * 1000 if len(above) else np.nan
    row["max_abs_mean_-15_to_5ms"] = float(np.abs(grand[:ac2.PRE + 20]).max())
    clock.append(row)
clock = pd.DataFrame(clock).set_index("neuron")
print("Fraction of variance explained by the tone (eta squared), after and before onset, with the null:")
print(clock.round(3).to_string())
print("\nWhole cell: post-onset eta2 above the largest of 20 permuted values in",
      int((clock.post_onset > clock.null_max).sum()), "of", len(clock), "neurons; pre-onset above it in",
      int((clock.pre_onset > clock.null_max).sum()))
lc = clock.dropna(subset=["lfp_post"])
print("LFP: post-onset above the largest permuted value in", int((lc.lfp_post > lc.lfp_null_max).sum()), "of", len(lc),
      "; pre-onset above it in", int((lc.lfp_pre > lc.lfp_null_max).sum()))

The tone sequence of the documentation is aligned with the traces: in every neuron the tone
explains the response after onset beyond the largest of 20 permuted labellings, and the mean
response rises above 1 mV only after onset (`onset_latency_ms`, printed above), with the mean
trace flat from 15 ms before to 5 ms after onset. The 100 ms before onset is also tone-dependent
beyond the null in most neurons; this is expected with a fixed tone order, since each tone is
always preceded by the same tone 500 ms earlier, whose slow after-effects fall in that window.
So this contrast cannot separate an offset from a response, and the alignment rests on the
onset latency and the post-onset tuning. The LFP and the whole-cell traces are cut from the same
sweeps at the same samples (the sweeps file holds both channels), so they share the clock.

In [ ]:
# Units and ranges of the variables used.
allr = np.concatenate([n["r"][ac2.stable(n)].ravel()[::50] for n in N.values()])
alll = np.concatenate([n["lfp"][ac2.stable(n)].ravel()[::50] for n in N.values() if ac2.has_lfp(n)])
print("r (mV, relative to the 15-ms baseline), 0.1 / 50 / 99.9 percentiles:", np.percentile(allr, [0.1, 50, 99.9]).round(2))
print("lfp (mV after the gain correction), 0.1 / 50 / 99.9 percentiles:", np.percentile(alll, [0.1, 50, 99.9]).round(3))
base = pd.Series({name: np.median(n["rbaseline"][ac2.stable(n)]) for name, n in N.items()})
print("Median recorded baseline per neuron (mV, uncorrected): %.1f to %.1f" % (base.min(), base.max()))
gains = info.loc[list(N), ["axogaincorrection_wholecell", "axogaincorrection_LFP"]]
print("Whole-cell gain corrections:", gains.axogaincorrection_wholecell.unique(),
      "; LFP gain corrections:", pd.Series(gains.axogaincorrection_LFP.astype(float)).value_counts(dropna=False).to_dict())
lfp_param = {name: n["params"].get("Axo_gain_correction_of_lfp") for name, n in N.items()}
mism = [name for name in N if not (np.isnan(float(lfp_param[name])) and np.isnan(float(info.loc[name, "axogaincorrection_LFP"])))
        and float(lfp_param[name]) != float(info.loc[name, "axogaincorrection_LFP"])]
print("Neurons whose LFP gain in cell_specific_parameters.m differs from neuron_recording_info.mat:", mism)
print("TTX recordings:", {k: (len(v["ep"]), v["params"]["first_trial"], v["params"]["last_trial"], v["gain"]) for k, v in D["ttx"].items()})

## The paper's procedure, under every reading

All claims on variability use the same per-tone values: PSP heights, q, and q after each stage of
Figure 6 (PSP-based culling, LFP-based culling, estimation with its overfitting correction, and
both cullings with estimation). They are computed here once per reading of the open details
(claims file header, details 1 to 7), and each claim's section reads them. The main reading:

- trials: the stable epoch of each neuron (`first_cycle` to `last_cycle`), without spike trials;
- spikes: the data set's per-neuron `spikethresh` above the trace's baseline (an assumption; see
  below), with the claims file's -20 mV threshold as a reading;
- peak of the mean PSP searched over the whole trace after onset (detail 1a), the culling window as a reading (1b);
- peak-to-trough height over the culling window (2a), the whole trace as a reading (2b);
- variance over N - 1 (3), N as a reading;
- negative scale factors removed in PSP-based culling too (4a), not removed as a reading (4b);
- the 3 mV and 15 mV criteria on the mean before culling (5a), after it as a reading (5b);
- overfitting correction per neuron, mean over 20 shuffles (7), per tone as a reading.

Heights are measured once on the full ensemble; when both cullings are applied, each is decided
on the full ensemble and a trial is kept when neither removes it (choices stated in `ac2.py`).

In [ ]:
READINGS = {
    "main": {},
    "spikes -20 mV (6, claims file)": {"spike_rule": "abs"},
    "peak in culling window (1b)": {"peak_window": "culling"},
    "peak-to-trough over trace (2b)": {"span": "trace"},
    "variance over N (3)": {"ddof": 0},
    "no negative-f rule in PSP culling (4b)": {"psp_negative": False},
}
SUMMARY_READINGS = {"criteria after culling (5b)": {"criteria": "after"},
                    "correction per tone (7)": {"per_tone_correction": True}}
t0 = time.time()
PT = {label: ac2.population(N, seed=2004, **opts) for label, opts in READINGS.items()}
S = {label: ac2.summarise(PT[label], N) for label in READINGS}
for label, opts in SUMMARY_READINGS.items():
    S[label] = ac2.summarise(PT["main"], N, **opts)
print(f"{len(S)} readings computed in {time.time() - t0:.0f} s")
main = S["main"]
lfp_neurons = main.index[main.lfp]

def mse(x, digits=2):
    x = pd.Series(x).dropna()
    return f"{x.mean():.{digits}f} +- {x.std(ddof=1) / np.sqrt(len(x)):.{digits}f} (n = {len(x)})"

overview = pd.DataFrame({label: {
    "<q> 33": mse(df.q), "<q> 29 LFP": mse(df.q[df.lfp]), "<q PSP-culled>": mse(df.q_psp),
    "<q LFP-culled>": mse(df.q_lfp[df.lfp]), "<q LFP-culled&est>": mse(df.q_lfp_est[df.lfp]),
    "<q PSP,LFP-culled&est>": mse(df.q_both_est)} for label, df in S.items()}).T
print(overview.to_string())

Spike trials. The claims file's header treats spike detection as a standard choice that "can
change only the few trials with spikes" (detail 6). It changes more than a few: the -20 mV rule
removes up to 73 trials of a neuron (`spikes_-20mV` in the data check) and lowers the population
<q> from about 4.6 to about 4.1 mV (table above), because in several neurons the largest PSPs
cross -20 mV without a spike. The data set gives each neuron its own threshold
(`spikethresh` in `cell_specific_parameters.m`, 28 to 55 mV above the trace's baseline), which
the TTX neurons' copies of the same file describe as the "height of threshold (mV) above
BASELINE/REST for spike detection/removal". Under step 1.6 of the protocol both readings are
shown for every claim, and the per-neuron threshold, the authors' own parameter, is adopted as
an assumption; outcomes that differ between the two are marked as resting on it.

## fig3a-reliable-neuron

> For the most reliable neuron, the responses to 20 presentations of a 32-kHz tone cluster
> tightly about the mean (q = 0.23 mV), and its variability averaged over the 26 tones with a
> mean PSP above 3 mV is 0.88 mV.

This section comes first because fig2-stereotyped-shape uses the neuron it identifies.

In [ ]:
def tone_table(label, name):
    return pd.DataFrame([{k: v for k, v in r.items() if np.isscalar(v)} for r in PT[label][name]]).set_index("tone")

for label in ("main", "spikes -20 mV (6, claims file)", "peak in culling window (1b)", "variance over N (3)"):
    df = S[label]
    best = df.q.idxmin()
    near = (df.q - 0.88).abs().idxmin()
    tt = tone_table(label, best)
    print(f"[{label}] lowest <q>tones: {best} <q> = {df.q[best]:.3f} mV over {df.n_tones[best]} tones;"
          f" neuron nearest 0.88 mV: {near} ({df.q[near]:.3f}); repetitions {int(cells.reps[best])};"
          f" q at 31,118 Hz {tt.q[31118]:.3f}, at 34,448 Hz {tt.q[34448]:.3f}")
EXAMPLE = S["main"].q.idxmin()
print("\nLowest five <q>tones (main):", S["main"].q.nsmallest(5).round(3).to_dict())
tt = tone_table("main", EXAMPLE)
print(tt[["n", "mean", "q", "peak_ms"]].round(3).to_string())

**Result: reproduced.** The neuron with the lowest <q>tones, 022801md01b, has <q>tones = 0.88 mV
over 26 tones with a mean PSP above 3 mV and 20 repetitions per tone, as in the paper, under the
main reading and with the -20 mV spike rule (no spike trials in this neuron). Its q for the
31,118-Hz tone is 0.229 mV (reading a of "32 kHz", the nearest tone); for 34,448 Hz (reading b)
it is 0.648 mV. The neuron and the 0.88 mV do not depend on the spike rule or the peak window;
with the variance over N, <q>tones is 0.832 mV and q at 31,118 Hz 0.217 mV, both within
tolerance.

## fig2-stereotyped-shape

> Tones evoke brief, short-latency PSPs in all neurons. In the example neuron the mean
> responses to the 32 tones peak 27 to 42 ms after onset with heights of 1.2 to 18 mV, and are
> similar in shape once normalized and aligned.

In [ ]:
def mean_traces(n):
    idx_ok = ac2.stable(n) & ~ac2.spikes(n, "cell")
    return {int(t): n["r"][idx_ok & (n["tone"] == t)].astype(float).mean(0) for t in ac2.TONE_ORDER}

for window, (lo, hi) in {"after onset (1a)": (ac2.PRE, ac2.TRACE), "culling window (1b)": (ac2.PRE, 400)}.items():
    mt = mean_traces(N[EXAMPLE])
    lat = {t: (lo + np.argmax(m[lo:hi]) - ac2.PRE) / ac2.FS * 1000 for t, m in mt.items()}
    hgt = {t: m[lo:hi].max() for t, m in mt.items()}
    print(f"[{window}] {EXAMPLE}: latency to peak {min(lat.values()):.2f} to {max(lat.values()):.2f} ms;"
          f" peak height {min(hgt.values()):.2f} to {max(hgt.values()):.2f} mV (32 tones)")
lat = pd.Series({t: (ac2.PRE + np.argmax(m[ac2.PRE:]) - ac2.PRE) / ac2.FS * 1000 for t, m in mt.items()})
hgt = pd.Series({t: m[ac2.PRE:].max() for t, m in mt.items()})
print("Latencies (ms) by tone:", lat.round(2).sort_index().to_dict())
print("Tones with latency outside 24.3-46.2 ms:", lat[(lat < 24.3) | (lat > 46.2)].round(2).to_dict())

# Shape: normalize each mean trace to its peak and align it on the peak; window = the culling
# window placed for a tone at the median latency.
med = ac2.PRE + int(round(np.median(lat.values) * ac2.FS / 1000))
before, after = med, 400 - med
aligned = []
for t, m in mt.items():
    pk = ac2.PRE + int(np.argmax(m[ac2.PRE:]))
    padded = np.pad(m, (before, after), mode="edge")       # edge values where the window runs off the trace
    aligned.append(padded[pk:pk + before + after] / m[pk])
aligned = np.array(aligned)
avg = aligned.mean(0)
cors = np.array([np.corrcoef(x, avg)[0, 1] for x in aligned])
print(f"Aligned traces: {len(aligned)} of 32, window {-before / 4:.2f} to {after / 4:.2f} ms around the peak; correlation with their average: median {np.median(cors):.3f},"
      f" min {cors.min():.3f}, 10th percentile {np.percentile(cors, 10):.3f}")
# Unaligned (no shift), for reference.
raw = np.array([m[:400] / m[ac2.PRE:].max() for m in mt.values()])
cr = np.array([np.corrcoef(x, raw.mean(0))[0, 1] for x in raw])
print(f"Without alignment: median correlation {np.median(cr):.3f}")

# All neurons: latency of the mean response to each neuron's largest tone.
big = []
for name, n in N.items():
    m = mean_traces(n)
    t_big = max(m, key=lambda t: m[t][ac2.PRE:].max())
    big.append({"neuron": name, "tone": t_big, "peak_mV": m[t_big][ac2.PRE:].max(),
                "latency_ms": np.argmax(m[t_big][ac2.PRE:]) / ac2.FS * 1000})
big = pd.DataFrame(big).set_index("neuron")
print(f"\nLargest tone per neuron: latency {big.latency_ms.min():.2f} to {big.latency_ms.max():.2f} ms "
      f"(median {big.latency_ms.median():.2f}); within 50 ms in {int((big.latency_ms <= 50).sum())} of {len(big)}")
print(big[big.latency_ms > 50].round(2).to_string())

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
tms = (np.arange(ac2.TRACE) - ac2.PRE) / ac2.FS * 1000
for m in mt.values():
    ax[0].plot(tms, m, lw=0.7)
for x in aligned:
    ax[1].plot((np.arange(before + after) - before) / ac2.FS * 1000, x, lw=0.7)
ax[0].set(xlabel="ms from onset", ylabel="mV", title=f"{EXAMPLE}: mean per tone (all within axes)")
ax[1].set(xlabel="ms from peak", ylabel="normalized", title="normalized and aligned")
plt.tight_layout()
plt.show()
print("Plotted ranges: mean traces %.2f to %.2f mV; normalized %.2f to %.2f" % (
    min(m.min() for m in mt.values()), max(m.max() for m in mt.values()), aligned.min(), aligned.max()))

**Result: partially reproduced.** In 022801md01b (20 trials per tone, the Figure 2 legend's 20
presentations) the mean responses to the 32 tones peak 26.5 to 41.5 ms after onset with heights
of 1.19 to 18.12 mV, within the widened ranges (24.3 to 46.2 ms; 1.08 to 19.8 mV), under both
peak windows. Normalized and aligned on their peaks, the 32 mean traces correlate with their
average with a median r of 0.991 (minimum 0.933), above the 0.9 the criterion sets. The "all
neurons" part does not hold under the claims file's 50-ms bound: the mean response to each
neuron's largest tone peaks within 50 ms of onset in 19 of 33 neurons (20.75 to 115.5 ms, median
45.25); the 14 others peak 53 to 115.5 ms after onset.

## fig3b-total-variability-distribution

> Across the 33 neurons the tone-averaged variability index ranges over more than an order of
> magnitude, from below 1 to above 10 mV; about a fifth (7/33) have low variability (1 to 2 mV)
> and about half (18/33) have high variability (4 to 12 mV).

In [ ]:
rows = []
for label, df in S.items():
    if label in SUMMARY_READINGS:
        continue
    q = df.q
    rows.append({"reading": label, "min": q.min(), "max": q.max(), "1<q<2": int(((q > 1) & (q < 2)).sum()),
                 "q<2": int((q < 2).sum()), "4<q<12": int(((q > 4) & (q < 12)).sum())})
print(pd.DataFrame(rows).set_index("reading").round(2).to_string())
print("\n<q>tones per neuron (main), sorted:", main.q.sort_values().round(2).tolist())
print("Neurons by anesthetic within 4<q<12 (main):", main[(main.q > 4) & (main.q < 12)].anesthetic.value_counts().to_dict())

**Result: reproduced** (resting on the spike-rule assumption). Under the main reading <q>tones
ranges from 0.88 to 11.21 mV (below 1 and above 10 mV). 18 neurons lie between 4 and 12 mV. 6
lie strictly between 1 and 2 mV, outside 7 +- 0.7; under the criterion's second reading of the
low count (q < 2 mV, which includes the 0.88 mV example neuron) it is 7. Both counts hold under
that reading. With the -20 mV spike rule the counts are 6 (7 below 2 mV) and 16, outside 18 +-
1.8, which would make the claim partially reproduced.

## total-variability-population

> The mean total variability index across neurons is 4.6 mV.

In [ ]:
for label, df in S.items():
    if label not in SUMMARY_READINGS:
        print(f"{label:42s} <q>neurons = {mse(df.q)}")
print("\nBy anesthetic (main):", main.groupby("anesthetic").q.agg(["mean", "count"]).round(2).to_dict("index"))

**Result: reproduced** (resting on the spike-rule assumption). <q>neurons is 4.58 +- 0.47 mV (SE,
n = 33), against 4.6 +- 0.5. With the -20 mV spike rule it is 4.05 mV, outside the tolerance
(4.1 to 5.1); with the peak searched in the culling window 4.22, and with the variance over N
4.29, both within it.

## lfp-culling-reduction

> LFP-based culling reduces the mean variability index by 35%, from 4.8 to 3.1 mV, across the
> 29 neurons with an LFP.

In [ ]:
print("Neurons with the LFP flag:", len(lfp_neurons))
for label, df in S.items():
    d = df[df.lfp]
    print(f"{label:42s} before {mse(d.q)}; after LFP-based culling {mse(d.q_lfp)};"
          f" fall {100 * (1 - d.q_lfp.mean() / d.q.mean()):.0f}%")
d = main[main.lfp]
print("\nTones per neuron before and after LFP-based culling (main):")
print(d[["n_tones", "n_tones_lfp", "q", "q_lfp"]].round(2).T.to_string())

**Result: reproduced** (resting on the spike-rule assumption). Over the 29 neurons with the LFP
flag, <q> is 4.80 +- 0.52 mV before LFP-based culling and 3.19 +- 0.43 mV after it (paper 4.8
+- 0.5 and 3.1 +- 0.1; both within tolerance), a fall of 33%. With the -20 mV spike rule the
values are 4.25 (outside 4.3 to 5.3) and 3.02 mV. All readings of details 2, 4 and 5 keep both
values within tolerance.

## fig5b-residual-lfp-psp-correlation

> After LFP-based culling, PSP height still correlates with LFP height across trials for many
> tones, as in the 60 unculled responses to one tone of the Figure 5 neuron.

In [ ]:
def residual_correlations(label):
    out = []
    for name in lfp_neurons:
        for r in PT[label][name]:
            if not ac2.included(r, "lfp"):
                continue
            k = r["keep_lfp"]
            h, s, lh = r["h"][k], r["s"][k], r["lfp_h"][k]
            rs, ps = stats.pearsonr(s, h)
            rl, pl = stats.pearsonr(lh, h)
            out.append({"neuron": name, "tone": r["tone"], "n": int(k.sum()), "r_scale": rs, "p_scale": ps,
                        "r_lfp_height": rl, "p_lfp_height": pl})
    return pd.DataFrame(out)

for label in ("main", "spikes -20 mV (6, claims file)", "peak in culling window (1b)"):
    rc = residual_correlations(label)
    print(f"[{label}] {len(rc)} tones of {rc.neuron.nunique()} neurons")
    for col in ("scale", "lfp_height"):
        r_, p_ = rc[f"r_{col}"], rc[f"p_{col}"]
        print(f"   LFP {col:10s}: positive {100 * (r_ > 0).mean():.0f}%, p < 0.05 {100 * (p_ < 0.05).mean():.0f}%"
              f" (positive and p < 0.05 {100 * ((r_ > 0) & (p_ < 0.05)).mean():.0f}%), median r {r_.median():.3f};"
              f" chance for p < 0.05 is 5%")
rc = residual_correlations("main")
FIG5 = cells.reps.idxmax()
print(f"\nFigure 5 neuron (63 repetitions): {FIG5}")
print(rc[rc.neuron == FIG5].round(3).to_string(index=False))
print("Tones of", FIG5, "with 60 trials left:", rc[(rc.neuron == FIG5) & (rc.n == 60)].tone.tolist())
per_neuron = rc.groupby("neuron").r_scale.median()
print("Neurons whose median residual correlation (scale factor) is positive:", int((per_neuron > 0).sum()), "of", len(per_neuron))

**Result: reproduced.** Over 396 included tones of the 29 neurons, after LFP-based culling, PSP
height correlates positively with the LFP scale factor in 81% of tones and with p < 0.05 in 32%
(30% positive and significant; 5% expected by chance), above the criterion's half and quarter.
With the LFP peak height in place of the scale factor, 75% and 26%. The Figure 5 neuron
(020701md01b, 63 repetitions) has two tones with 60 trials left after culling (31,118 and 5,527
Hz), with r = 0.70 and 0.65 to the scale factor (p < 0.001).

## fig7a-lfp-culling-estimation

> LFP-based culling and estimation together reduce the variability index by 40% on average, to
> 2.9 mV across 29 neurons, reduce it in 26 of 29 neurons, by as much as 71%, and have little
> effect when total variability is already small.

In [ ]:
rows = []
for label, df in S.items():
    d = df[df.lfp].dropna(subset=["q_lfp_est"])
    red = 1 - d.q_lfp_est / d.q
    rows.append({"reading": label, "n": len(d), "<q LFP-culled&est>": mse(d.q_lfp_est),
                 "reduced": int((red > 0).sum()), "largest %": 100 * red.max(),
                 "(a) 1 - est/total %": 100 * (1 - d.q_lfp_est.mean() / d.q.mean()),
                 "(b) mean of reductions %": 100 * red.mean()})
print(pd.DataFrame(rows).set_index("reading").round(1).to_string())
d = main[main.lfp]
show = pd.DataFrame({"total": d.q, "LFP-culled&est": d.q_lfp_est, "reduction %": 100 * (1 - d.q_lfp_est / d.q),
                     "uncorrected": d.q_lfp_est_uncorrected, "tones": d.n_tones_lfp})
print(show.sort_values("total").round(2).to_string())
print("\nNeurons with total <q>tones below 1.5 mV (shown, not scored):")
print(show[show.total < 1.5].round(2).to_string())
print("Largest reductions (main):", show["reduction %"].nlargest(3).round(1).to_dict())

**Result: partially reproduced** (resting on the spike-rule assumption). <q LFP-culled&estimation>
is 2.68 +- 0.32 mV over 29 neurons (2.9 +- 0.3), and the procedures reduce <q>tones in 26 of 29
neurons, as in the paper. The average reduction is 44.1% by reading (a) (just outside 36 to 44%)
and 40.2% by reading (b), so the 40% holds under (b). The largest reduction is 80.1%
(030301md03b), outside 71 +- 7.1%. In the four neurons with total <q>tones below 1.5 mV the
procedures lower it by 8.5 to 49.6% (to 0.61 to 1.13 mV); shown, not scored. With the -20 mV
spike rule the value is 2.53 mV (outside), with 28 neurons reduced and a largest reduction of
72.4%.

## psp-culling-reduction

> PSP-based culling alone reduces the mean variability index to 2.2 mV across 33 neurons, and
> leaves it at most about 3 mV in 32 of 33 neurons.

In [ ]:
for label, df in S.items():
    d = df.dropna(subset=["q_psp"])
    print(f"{label:42s} <q PSP-culled> {mse(d.q_psp)}; at most 3.3 mV in {int((d.q_psp <= 3.3).sum())} of {len(d)};"
          f" at most 3.0 mV in {int((d.q_psp <= 3.0).sum())}")
print("\nPer neuron (main):", main.q_psp.sort_values().round(2).to_dict())

**Result: partially reproduced** (resting on the spike-rule assumption). <q PSP-culled> is 2.19 +-
0.31 mV over 33 neurons (2.2 +- 0.3), but 30 of 33 neurons are at or below 3.3 mV, outside 32
+- 1; the three above are 3.38, 3.60 and 10.98 mV. With the peak searched in the culling window
(detail 1b) or the variance over N (detail 3) the count is 31 or 32 with the mean in tolerance;
with the -20 mV spike rule 2.29 mV and 29 neurons.

## fig4c-psp-culling-can-increase

> PSP-based culling reduces the tone-averaged variability for all but one neuron, but increases
> the variability for at least one tone in half (18/33) of the neurons; in the Figure 4 neuron,
> it raises q for a 9.2-kHz tone from 0.47 to 0.57 mV.

In [ ]:
rows = []
for label, df in S.items():
    pt = PT[label] if label in PT else PT["main"]
    crit = SUMMARY_READINGS.get(label, {}).get("criteria", "before")
    up = 0
    for name, tones in pt.items():
        if any(ac2.included(r, None, crit) and ac2.included(r, "psp", crit) and r["q_psp"] > r["q"] for r in tones):
            up += 1
    d = df.dropna(subset=["q_psp"])
    rows.append({"reading": label, "<q> falls": int((d.q_psp < d.q).sum()), "of": len(d),
                 "neurons with a tone whose q rises": up})
print(pd.DataFrame(rows).set_index("reading").to_string())
print("Neurons whose <q>tones rises (main):", main[main.q_psp >= main.q][["q", "q_psp"]].round(2).to_dict("index"))

# Figure 4 neuron: one tone where PSP-based culling removes one trace and LFP-based culling removes
# that trace and three others (Figure 4B legend).
cand = []
for name in lfp_neurons:
    for r in PT["main"][name]:
        if "keep_lfp" not in r:
            continue
        rp, rl = ~r["keep_psp"], ~r["keep_lfp"]
        if rp.sum() == 1 and rl.sum() == 4 and (rp & rl).sum() == 1:
            cand.append({"neuron": name, "tone": r["tone"], "n": r["n"]})
cand = pd.DataFrame(cand)
print(f"\nTones fitting the Figure 4B legend: {len(cand)} in {cand.neuron.nunique() if len(cand) else 0} neurons",
      cand.groupby("neuron").tone.apply(list).to_dict() if len(cand) else "")
pairs = []
for name, tones in PT["main"].items():
    r = next(r for r in tones if r["tone"] == 9188)
    if "q_psp" in r and "q" in r:
        pairs.append({"neuron": name, "q_9188": r["q"], "q_9188_psp": r["q_psp"], "n": r["n"], "n_psp": r["n_psp"],
                      "mean": r["mean"], "fits_4B": name in set(cand.neuron) if len(cand) else False})
pairs = pd.DataFrame(pairs).set_index("neuron")
pairs["distance"] = np.hypot(pairs.q_9188 - 0.47, pairs.q_9188_psp - 0.57)
print("9,188-Hz tone, closest five to (0.47, 0.57):")
print(pairs.nsmallest(5, "distance").round(3).to_string())
print("9,188-Hz tone in the neurons fitting Figure 4B:")
print(pairs[pairs.fits_4B].round(3).to_string())

**Result: partially reproduced.** PSP-based culling lowers <q>tones in 32 of 33 neurons (the
exception is 021401md02, whose one remaining tone rises from 5.19 to 10.98 mV), as the paper
says. q rises for at least one tone in 16 of 33 neurons, outside 18 +- 1.8 (17 with the -20 mV
spike rule, within it; 20 without the negative scale factor rule in PSP culling). The Figure 4
neuron is not identified: 11 tones in 8 neurons fit the Figure 4B legend, and in none of them
does the 9,188-Hz tone give 0.47 and 0.57 mV; the closest 9,188-Hz pair in any neuron is 0.470
to 0.502 mV (030301md03b).

## fig7b-private-best-estimate

> PSP-based culling followed by LFP-based culling and estimation reduces variability by as much
> as 85%, to a best estimate of private variability of 1.6 mV across 28 neurons, comparable and
> low for all neurons.

In [ ]:
rows = []
for label, df in S.items():
    d = df.dropna(subset=["q_both_est"])
    red = 1 - d.q_both_est / d.q
    rows.append({"reading": label, "<q best>": mse(d.q_both_est), "neurons": len(d), "largest reduction %": 100 * red.max(),
                 "min": d.q_both_est.min(), "max": d.q_both_est.max(), "SD": d.q_both_est.std(ddof=1)})
print(pd.DataFrame(rows).set_index("reading").round(2).to_string())
d = main.dropna(subset=["q_both_est"])
print("\nNeuron with an LFP but no estimate (main):", list(set(lfp_neurons) - set(d.index)))
print("Best estimate per neuron (main):", d.q_both_est.sort_values().round(2).to_dict())
print("Tones kept per neuron:", d.n_tones_both.to_dict())
print("Best estimate within 0.9-3.3 mV:", int(d.q_both_est.between(0.9, 3.3).sum()), "of", len(d),
      "; total <q>tones SD across these neurons %.2f, best estimate SD %.2f" % (d.q.std(ddof=1), d.q_both_est.std(ddof=1)))

**Result: reproduced.** The best estimate of private variability, PSP-based culling then
LFP-based culling and estimation, is 1.51 +- 0.21 mV over 28 neurons (1.6 +- 0.2; 28 exact), and
the largest reduction from total variability is 87.8% (85 +- 8.5). Under every reading the mean
is 1.39 to 1.75 mV. The values are low for most neurons (0.30 to 2.85 mV) except 110900md05 (5.94
mV, two tones); their SD across neurons is 1.09 mV against 2.85 mV for total variability.

## culling-controls

> Random culling of as many traces as LFP-based culling removes leaves the variability index
> unchanged (4.5 against 4.6 mV), while matching the PSP-based or both cullings lowers it (3.67
> and 2.98 mV) because tones with fewer than 10 non-aberrant trials drop out; keeping only those
> tones without culling gives 2.9 mV. PSP-based culling removes 29% of trials and LFP-based
> culling 38%.

In [ ]:
def removed_mask(r, kind):
    if kind == "psp":
        return ~r["keep_psp"]
    if "keep_lfp" not in r:
        return None
    return ~r["keep_lfp"] if kind == "lfp" else ~(r["keep_psp"] & r["keep_lfp"])

def random_control(pt, kind, names, seed, min_mean=ac2.MIN_MEAN, n_sims=20, ddof=1):
    """Remove at random, per neuron and tone, as many trials as the named culling removed; then
    apply the tone criteria (at least 10 trials left, mean of all trials above min_mean) and take <q>neurons."""
    rng = np.random.default_rng(seed)
    sims = []
    for _ in range(n_sims):
        per_neuron = []
        for name in names:
            qs = []
            for r in pt[name]:
                if r["n"] < ac2.MIN_TRIALS or "mean" not in r or not r["mean"] > min_mean:
                    continue
                rem = removed_mask(r, kind)
                k = int(rem.sum()) if rem is not None else 0
                keep = rng.permutation(r["n"])[k:]
                if len(keep) >= ac2.MIN_TRIALS:
                    qs.append(ac2.q_index(r["h"][keep], ddof))
            if qs:
                per_neuron.append(np.mean(qs))
        sims.append((np.mean(per_neuron), len(per_neuron)))
    v = np.array([s[0] for s in sims])
    return {"mean": v.mean(), "SD over sims": v.std(ddof=1), "SE over sims": v.std(ddof=1) / np.sqrt(n_sims),
            "neurons": int(np.median([s[1] for s in sims]))}

all_names, lfp_names = list(N), list(lfp_neurons)
def controls(label, min_mean=ac2.MIN_MEAN):
    pt = PT[label]
    out = {
        "LFP-matched, 29 neurons": random_control(pt, "lfp", lfp_names, 11, min_mean),
        "LFP-matched, 33 (4 unculled)": random_control(pt, "lfp", all_names, 12, min_mean),
        "PSP-matched, 33 neurons": random_control(pt, "psp", all_names, 13, min_mean),
        "PSP-matched, 29 neurons": random_control(pt, "psp", lfp_names, 14, min_mean),
        "both-matched (union), 29 neurons": random_control(pt, "both", lfp_names, 15, min_mean),
    }
    return pd.DataFrame(out).T

for label in ("main", "spikes -20 mV (6, claims file)"):
    print(f"[{label}]")
    print(controls(label).round(3).to_string())
    df = S[label]
    print(f"Second control: unculled <q>tones over tones that keep 10 trials after both cullings: {mse(df.q_on_both_tones)}")

def fraction_removed(pt, kind, names):
    pooled_rem = pooled_all = 0
    per = []
    for name in names:
        rem_n = all_n = 0
        for r in pt[name]:
            if not ac2.included(r):
                continue
            rem = removed_mask(r, kind)
            rem_n += int(rem.sum())
            all_n += r["n"]
        pooled_rem += rem_n
        pooled_all += all_n
        if all_n:
            per.append(rem_n / all_n)
    return 100 * pooled_rem / pooled_all, 100 * np.mean(per)

for label in ("main", "spikes -20 mV (6, claims file)", "peak-to-trough over trace (2b)", "no negative-f rule in PSP culling (4b)"):
    pt = PT[label]
    f_psp33 = fraction_removed(pt, "psp", all_names)
    f_psp29 = fraction_removed(pt, "psp", lfp_names)
    f_lfp = fraction_removed(pt, "lfp", lfp_names)
    print(f"[{label}] trials removed, pooled / mean of neurons: PSP-based {f_psp33[0]:.1f}% / {f_psp33[1]:.1f}% (33 neurons),"
          f" {f_psp29[0]:.1f}% / {f_psp29[1]:.1f}% (29); LFP-based {f_lfp[0]:.1f}% / {f_lfp[1]:.1f}% (29)")

**Result: not reproduced.** Two of the four q values and one of the two fractions match.
Random culling matched to both cullings gives 2.988 mV (2.98 +- 0.02) and the second control
2.95 +- 0.32 mV over 28 neurons (2.9 +- 0.3). Random culling matched to LFP-based culling gives
3.82 mV over the 29 LFP neurons (3.80 over 33), not the 4.5 +- 0.1 mV of the paper: removing 38%
of trials at random drops tones below 10 trials and lowers <q> by about 0.8 mV, so the
undersampling the control tests for is present here. Matched to PSP-based culling it gives 3.737
mV over 33 neurons (3.67 +- 0.02; the miss, 0.067, is more than three SEs) and 3.86 over 29.
PSP-based culling removes 24.1% of the trials of included tones (pooled; 19.2% as a mean of
neurons), outside 29 +- 2.9%, and LFP-based culling 37.8% (38 +- 3.8).

## fig7d-large-responses

> For tones with mean PSPs of at least 15 mV, total variability is lower (2.9 mV, 19 neurons)
> than for all tones, but the private variability is the same as for all tones (best estimate
> 1.5 mV, LFP-based bound 1.9 mV, 16 neurons).

In [ ]:
LARGE = ac2.LARGE_MEAN - 1e-9          # "at least 15 mV": the criterion is "> min_mean"
SL = {}
for label in READINGS:
    SL[label] = ac2.summarise(PT[label], N, min_mean=LARGE)
SL["criteria after culling (5b)"] = ac2.summarise(PT["main"], N, criteria="after", min_mean=LARGE)
rows = []
for label, df in SL.items():
    tot = df.q.dropna()
    rows.append({"reading": label, "<q> total": mse(tot), "<q LFP-culled&est>": mse(df.q_lfp_est[df.lfp]),
                 "<q best>": mse(df.q_both_est)})
print(pd.DataFrame(rows).set_index("reading").to_string())
dl = SL["main"]
print("\nNeurons with a tone of at least 15 mV (main):", int(dl.q.notna().sum()), list(dl.index[dl.q.notna()]))
print("Neurons with a best estimate on such tones:", list(dl.index[dl.q_both_est.notna()]))
print("Tones of at least 15 mV per neuron:", dl.n_tones[dl.n_tones > 0].to_dict())

**Result: reproduced.** For tones with a mean PSP of at least 15 mV, total <q> is 2.92 +- 0.36 mV
over 19 neurons (2.9 +- 0.4; 19 exact), the LFP-based bound 1.62 +- 0.23 mV (1.9 +- 0.3) and the
best estimate 1.38 +- 0.21 mV (1.5 +- 0.2). Both private values come from 14 neurons, not the
paper's 16; with the 3 mV and 15 mV criteria applied after culling (detail 5b) there are 15 and
16, with values 1.80 and 1.51 mV. With the -20 mV spike rule total <q> is 2.51 mV, also within.

## large-responses-controls

> For tones of at least 15 mV, random culling and keeping only tones that survive both cullings
> give 3.2 mV, so the private estimate of 1.5 mV is not an undersampling effect.

In [ ]:
for label in ("main", "spikes -20 mV (6, claims file)"):
    c = random_control(PT[label], "both", lfp_names, 16, LARGE)
    print(f"[{label}] random culling matched to both, tones of at least 15 mV: {c['mean']:.3f} mV"
          f" (SE over 20 simulations {c['SE over sims']:.3f}, SD {c['SD over sims']:.3f}, {c['neurons']} neurons)")
    print(f"[{label}] second control, unculled <q> over tones of at least 15 mV that keep 10 trials after both cullings:"
          f" {mse(SL[label].q_on_both_tones)}")

**Result: not reproduced.** For tones of at least 15 mV, random culling matched to both cullings
gives 2.43 mV (SE over 20 simulations 0.03) and the second control 2.40 +- 0.39 mV over 14
neurons, against 3.20 +- 0.05 and 3.2 +- 0.5. The two controls agree with each other and are
above the 1.38 mV best estimate, as the paper argues, but both values are outside tolerance
(with the -20 mV spike rule, 2.07 and 2.04 mV).

## cv-large-responses

> In coefficient of variation, both the total and the private variability of the large (at least
> 15 mV) responses are lower than for the full data set (total 0.37 against 0.73; private 0.27
> against 0.41, significantly lower), unlike the variability index.

In [ ]:
for label in ("main", "spikes -20 mV (6, claims file)", "variance over N (3)"):
    a, l = S[label], SL[label]
    pa, pl = a.cv_both_est.dropna(), l.cv_both_est.dropna()
    print(f"[{label}] <CV> total, >= 15 mV: {mse(l.cv, 3)}; all tones: {mse(a.cv, 3)}")
    print(f"[{label}] <CV> private, >= 15 mV: {mse(pl, 3)}; all tones: {mse(pa, 3)}")
    t, p = stats.ttest_ind(pl, pa)
    common = pl.index.intersection(pa.index)
    tp, pp = stats.ttest_rel(pl[common], pa[common])
    print(f"   (a) Student's t-test, {len(pl)} against {len(pa)} neurons: t = {t:.2f}, p = {p:.4f}")
    print(f"   (b) paired t-test over {len(common)} neurons: t = {tp:.2f}, p = {pp:.4f};"
          f" mean difference {np.mean(pl[common] - pa[common]):.3f}")

**Result: partially reproduced.** Total CV is 0.371 +- 0.027 for tones of at least 15 mV (19
neurons) and 0.734 +- 0.042 for all tones (33), matching 0.37 and 0.73. Private CV is 0.259 +-
0.021 (14 neurons) against 0.385 +- 0.031 (28), just outside 0.27 +- 0.01 and 0.41 +- 0.02.
The private CV of large responses is significantly lower under both readings of the test
(Student's t, p = 0.0085; paired over 14 neurons, p < 0.0001). The criterion's reproduced
outcome needs all four values; two miss by 0.001 and 0.005 beyond the tolerance, so the claim
holds in part. With the -20 mV spike rule the private large-response CV is 0.263 (within) and
the full-set private CV 0.386 (outside).

## lfp-cv-not-different

> The LFP recorded with the more variable neurons is not more variable than that recorded with
> the less variable neurons (CV 0.66 against 0.74, P = 0.5), so differences in total variability
> are not explained by gross state differences seen in the LFP.

In [ ]:
def lfp_cv(label, measure):
    out = {}
    for name in lfp_neurons:
        vals = []
        for r in PT[label][name]:
            if not ac2.included(r):
                continue
            x = r["lfp_h"] if measure == "height" else r["s"]
            vals.append(np.std(x, ddof=1) / np.mean(x))
        out[name] = np.mean(vals)
    return pd.Series(out)

for label in ("main", "spikes -20 mV (6, claims file)"):
    q29 = S[label].q[lfp_neurons].sort_values(ascending=False)
    more, less = q29.index[:14], q29.index[14:]
    for measure in ("height", "scale factor"):
        cv = lfp_cv(label, measure)
        t, p = stats.ttest_ind(cv[more], cv[less])
        print(f"[{label}; LFP {measure}] more variable (14): {mse(cv[more])}; less variable (15): {mse(cv[less])};"
              f" Student's t = {t:.2f}, p = {p:.3f}; difference {cv[more].mean() - cv[less].mean():.3f}")
cvh = lfp_cv("main", "height")
print("\nLFP CV per neuron (height, main):", cvh.round(2).to_dict())
print("Mean LFP height sign check: tones whose mean LFP height is negative:",
      sum(np.mean(r["lfp_h"]) < 0 for name in lfp_neurons for r in PT["main"][name] if ac2.included(r)))

**Result: reproduced.** Ranking the 29 LFP neurons by total <q>tones, the LFP height CV is 0.61
+- 0.05 for the 14 more variable and 0.57 +- 0.06 for the 15 less variable neurons (paper 0.66
+- 0.06 and 0.74 +- 0.09); Student's t-test finds no difference (t = 0.43, p = 0.674;
difference 0.035). The less variable group's CV is below the paper's tolerance. The LFP scale
factor CV gives 0.89 against 0.77 (p = 0.350), and the -20 mV spike rule p = 0.366. Unlike the
paper's slight opposite trend, the more variable neurons have the slightly higher LFP CV here.

## anesthesia-comparison

> Total variability does not differ significantly between the three anesthesia protocols
> (pentobarbital 3.7, pentobarbital with diazepam 6.1, urethane 5.0 mV), and private variability
> differs slightly but significantly between pentobarbital alone (1.1 mV) and urethane (2.2 mV).

In [ ]:
for label in ("main", "spikes -20 mV (6, claims file)"):
    df = S[label]
    print(f"[{label}]")
    for an in ("P", "P_with_Diaz", "U"):
        g = df[df.anesthetic == an]
        print(f"   {ac2.ANESTHETIC[an]:28s} total {mse(g.q)}; private {mse(g.q_both_est)}")
    groups = [df.q[df.anesthetic == an] for an in ("P", "P_with_Diaz", "U")]
    f, pf = stats.f_oneway(*groups)
    h, ph = stats.kruskal(*groups)
    print(f"   total: one-way ANOVA F = {f:.2f}, p = {pf:.3f}; Kruskal-Wallis H = {h:.2f}, p = {ph:.3f}")
    pp, pu = df.q_both_est[df.anesthetic == "P"].dropna(), df.q_both_est[df.anesthetic == "U"].dropna()
    t, pt_ = stats.ttest_ind(pp, pu)
    u, pmw = stats.mannwhitneyu(pp, pu)
    print(f"   private, P alone ({len(pp)}) against urethane ({len(pu)}): Student's t = {t:.2f}, p = {pt_:.4f};"
          f" Mann-Whitney U = {u:.0f}, p = {pmw:.4f}; difference {pu.mean() - pp.mean():.2f} mV")

**Result: partially reproduced** (resting on the spike-rule assumption). Total <q> is 3.69 +- 0.70
(pentobarbital, 13), 6.15 +- 2.81 (with diazepam, 3) and 4.99 +- 0.58 mV (urethane, 17), all
within tolerance; neither the one-way ANOVA (p = 0.251) nor Kruskal-Wallis (p = 0.354) finds a
difference. The private best estimate is 0.98 +- 0.13 (12), 1.24 +- 0.34 (3) and 2.06 +- 0.37
mV (13), with the paper's group sizes (12, 3, 13); pentobarbital alone and urethane differ
(Student's t p = 0.014; Mann-Whitney p = 0.0036). The pentobarbital private value, 0.98 mV, is
0.02 below the tolerance of 1.1 +- 0.1, so not every value holds. With the -20 mV rule it is
1.02 mV (within), but the urethane total, 4.29 mV, falls outside.

## fig8-spontaneous-veto

> The size of spontaneous activity before the stimulus is anticorrelated with the evoked PSP, in
> the LFP (mean correlation -0.14, 29 neurons) and in the whole-cell record (-0.22, 33 neurons),
> the whole-cell correlation being negative in 30 of 33 neurons.

The 100 ms before each tone come from the sweeps (`pre_wc`, `pre_lfp`). For the first tone of a
sweep this window overlaps the prepulse in neurons whose prepulse ends after 100 ms; those trials
are left out (counted below).

In [ ]:
def spont_size(x, channel, reading):
    if reading == "a":
        return x.max(1) - x.min(1)
    dev = x - x.mean(1, keepdims=True)
    return dev.max(1) if channel == "wc" else np.abs(dev).max(1)

def veto(label, channel, size_reading, corr_reading):
    out, dropped = {}, 0
    names = all_names if channel == "wc" else lfp_names
    for name in names:
        n = N[name]
        pre = n["pre_wc"] if channel == "wc" else n["pre_lfp"]
        per_tone, pooled_x, pooled_h = [], [], []
        for r in PT[label][name]:
            if not ac2.included(r):
                continue
            ok = ~n["prestim_overlaps_prepulse"][r["trials"]]
            dropped += int((~ok).sum())
            x = spont_size(pre[r["trials"]].astype(float), channel, size_reading)[ok]
            h = r["h"][ok]
            if len(h) >= 3 and np.ptp(x) > 0:
                per_tone.append(stats.pearsonr(x, h)[0])
                pooled_x.append(x)
                pooled_h.append(h / h.mean())
        if per_tone:
            out[name] = np.mean(per_tone) if corr_reading == "a" else stats.pearsonr(
                np.concatenate(pooled_x), np.concatenate(pooled_h))[0]
    return pd.Series(out), dropped

rows = []
for label in ("main", "spikes -20 mV (6, claims file)"):
    for sr in ("a", "b"):
        for cr in ("a", "b"):
            wc, dropped = veto(label, "wc", sr, cr)
            lf, _ = veto(label, "lfp", sr, cr)
            rows.append({"reading": label, "size": sr, "correlation": cr,
                         "LFP mean r": lf.mean(), "LFP SE": lf.std(ddof=1) / np.sqrt(len(lf)), "LFP n": len(lf),
                         "LFP negative": int((lf < 0).sum()),
                         "WC mean r": wc.mean(), "WC SE": wc.std(ddof=1) / np.sqrt(len(wc)), "WC n": len(wc),
                         "WC negative": int((wc < 0).sum()), "trials left out (prepulse)": dropped})
print(pd.DataFrame(rows).round(3).to_string(index=False))
wc, _ = veto("main", "wc", "a", "a")
print("\nWhole-cell correlation per neuron (main, readings a, a):", wc.round(3).to_dict())
print(f"Null: with PSP heights shuffled within tones the correlation is 0 in expectation; per-tone Pearson r"
      f" averaged over tones of {np.mean([len(r['h']) for r in PT['main'][FIG5] if ac2.included(r)]):.0f} trials (Figure 5 neuron).")
print("Figure 5 neuron:", FIG5, "whole-cell r (a, a) = %.3f" % wc[FIG5])

**Result: reproduced.** Under readings (a) size as maximum minus minimum and (b) one correlation
over all trials with heights normalized by the tone's mean, the mean correlation is -0.155 +-
0.028 for the LFP (29 neurons; -0.14 +- 0.02) and -0.193 +- 0.031 for the whole cell (33;
-0.22 +- 0.03), negative in 29 of 33 neurons (30 +- 3). Under readings (b, b) -0.149, -0.210 and
28 also hold. Under (a, a) the LFP mean, -0.168, is outside its tolerance. All eight
combinations, with either spike rule, give negative means for both channels (-0.149 to -0.169
LFP, -0.184 to -0.238 whole cell). 257 trials whose 100-ms window overlaps the prepulse are left
out.

## overfitting-correction-example

> For the Figure 5 neuron, the shuffle control attributes 0.02 mV to overfitting of the linear
> estimator, and the corrected value after both cullings and estimation is 0.57 mV.

In [ ]:
for label in ("main", "spikes -20 mV (6, claims file)", "peak-to-trough over trace (2b)", "no negative-f rule in PSP culling (4b)"):
    r = S[label].loc[FIG5]
    print(f"[{label}] {FIG5}: <q LFP-culled,PSP-culled> {r.q_both:.3f}; shuffled estimation {r.q_both_shuf:.3f};"
          f" overfitting {r.q_both - r.q_both_shuf:.3f}; estimation uncorrected {r.q_both_est_uncorrected:.3f};"
          f" corrected {r.q_both_est:.3f}; tones {int(r.n_tones_both)}")
r = S["correction per tone (7)"].loc[FIG5]
print(f"[per-tone correction (7)] corrected {r.q_both_est:.3f}")
# The shuffle reduction for a regression with one predictor is about q / (N - 1) per tone.
tones = [t for t in PT["main"][FIG5] if ac2.included(t, "both")]
print("Trials left per tone after both cullings:", [int(t["n_both"]) for t in tones])

**Result: reproduced.** In the Figure 5 neuron (020701md01b), after both cullings <q>tones is
0.759 mV (0.76), the shuffled estimation gives 0.746 mV (0.74), the estimation 0.525 mV
uncorrected (0.55) and 0.538 mV corrected (0.57); all four within 10%. The overfitting share is
0.013 mV against the paper's 0.02. The values are the same with the -20 mV spike rule and the
per-tone correction.

## private-shared-ranges

> The private component of the variability is small, about 1 to 3 mV, and relatively
> homogeneous across neurons, while the shared component is often much larger and more
> heterogeneous, ranging from about 0 to 10 mV.

In [ ]:
for label in ("main", "spikes -20 mV (6, claims file)"):
    d = S[label].dropna(subset=["q_both_est"])
    priv = d.q_both_est
    for total_reading, tot in (("all included tones", d.q), ("tones kept for the private estimate", d.q_on_both_tones)):
        shared = tot - priv
        print(f"[{label}; total over {total_reading}] private 0.9-3.3 mV in {int(priv.between(0.9, 3.3).sum())} of {len(priv)}"
              f" ({100 * priv.between(0.9, 3.3).mean():.0f}%), range {priv.min():.2f} to {priv.max():.2f}, SD {priv.std(ddof=1):.2f};"
              f" shared range {shared.min():.2f} to {shared.max():.2f} (within -1 to 11: {bool(shared.between(-1, 11).all())}),"
              f" SD {shared.std(ddof=1):.2f}")
d = S["main"].dropna(subset=["q_both_est"])
print("Private values outside 0.9-3.3 mV (main):", d.q_both_est[~d.q_both_est.between(0.9, 3.3)].round(2).to_dict())

**Result: partially reproduced.** Two of the three parts hold. Shared variability (total minus
private) ranges from -0.84 to 9.25 mV with total over all included tones, and 0.10 to 5.30 mV
with total over the tones kept for the private estimate, both within -1 to 11 mV, and its SD
across neurons (2.71 or 1.32 mV) exceeds that of private variability (1.09 mV). But only 18 of
28 neurons (64%) have a private value between 0.9 and 3.3 mV, short of the 80% the criterion
sets: nine are below 0.9 mV (0.30 to 0.89) and one is above (5.94 mV).

## quantal-comparison

> In 14 of 33 neurons at least one tone elicits responses whose variability index is within a
> factor of 2 of the 0.38 mV expected from stochastic release alone.

In [ ]:
rows = []
for label in ("main", "spikes -20 mV (6, claims file)", "peak in culling window (1b)", "variance over N (3)"):
    for ref in (0.38, 0.39):
        qs = {name: [r["q"] for r in tones if ac2.included(r)] for name, tones in PT[label].items()}
        rows.append({"reading": label, "reference": ref,
                     "q <= 2 x ref": sum(any(q <= 2 * ref for q in v) for v in qs.values()),
                     "ref/2 <= q <= 2 x ref": sum(any(ref / 2 <= q <= 2 * ref for q in v) for v in qs.values())})
print(pd.DataFrame(rows).to_string(index=False))
mins = pd.Series({name: min(r["q"] for r in tones if ac2.included(r)) for name, tones in PT["main"].items()})
print("Smallest included-tone q per neuron (main), sorted:", mins.sort_values().round(2).tolist())

**Result: reproduced.** 14 of 33 neurons have at least one included tone with q within a factor of
2 of 0.38 mV, under both readings of "within a factor of 2" (the smallest q is 0.14 mV, but that
neuron has another tone within range). With 0.39 mV the count is 15, also within 14 +- 1.4. With
the -20 mV spike rule it is 15 and 16, with the variance over N 17 and 20.

## mepsp-size

> Miniature EPSPs recorded in vivo after TTX have a mean size of 0.39 mV with a CV of 0.36
> across 6 neurons, consistent with in vitro estimates.

The paper gives no detection method. Two standard detectors are run (claims file readings a and
b), each at several thresholds: (a) template matching (Clements and Bekkers 1997; template rise
0.5 ms, decay 5 ms), threshold on the ratio of the fitted scale to its standard error; (b) the
derivative of the trace smoothed over 1 ms, threshold in robust SDs of the derivative. Size is
the peak in the 10 ms after onset minus the mean of the 2 ms before it. Sweeps are those of each
recording's stable epoch (`first_trial` to `last_trial`), without their first 0.3 s (prepulse).
One recording, 030401md02h, is divided by its documented gain of 4.

In [ ]:
rows = []
for name, rec in D["ttx"].items():
    fs = 1.0 / rec["dx"]
    sweeps = ac2.mini_sweeps(rec)
    noise = np.median([np.median(np.abs(np.diff(ac2._smooth(s, fs)))) / 0.6745 for s in sweeps])
    for det, ks in (("template (a)", (3.0, 4.0, 5.0)), ("derivative (b)", (3.0, 4.0, 5.0))):
        for k in ks:
            f = ac2.minis_template if det.startswith("template") else ac2.minis_derivative
            amps = np.concatenate([f(s, fs, k) for s in sweeps])
            amps = amps[amps > 0]
            rows.append({"neuron": name, "detector": det, "k": k, "seconds": sum(len(s) for s in sweeps) / fs,
                         "events": len(amps), "rate_hz": len(amps) / (sum(len(s) for s in sweeps) / fs),
                         "mean_mV": amps.mean() if len(amps) else np.nan,
                         "cv": amps.std(ddof=1) / amps.mean() if len(amps) > 1 else np.nan,
                         "median_mV": np.median(amps) if len(amps) else np.nan})
minis = pd.DataFrame(rows)
print(minis.round(3).to_string(index=False))
summ = minis.groupby(["detector", "k"]).agg(mean_size=("mean_mV", "mean"), se_size=("mean_mV", lambda x: x.std(ddof=1) / np.sqrt(len(x))),
                                             mean_cv=("cv", "mean"), se_cv=("cv", lambda x: x.std(ddof=1) / np.sqrt(len(x))),
                                             neurons=("mean_mV", "count"))
print("\nMean over the 6 neurons (paper: 0.39 +- 0.08 mV, CV 0.36 +- 0.03):")
print(summ.round(3).to_string())

**Result: blocked (methods underspecified).** The mean mEPSP size and its CV depend on the detector
and its threshold, which the paper does not give: over the six recordings the mean size is 0.168
to 0.174 mV with template matching at 3 to 5 SE (CV 0.66 to 0.75) and 0.338 to 0.476 mV with
the derivative detector at 3 to 5 robust SDs (CV 0.43 to 0.47), against 0.39 +- 0.08 mV and CV
0.36 +- 0.03. The derivative detector at 4 SDs gives a mean size within tolerance (0.412 mV)
but no setting gives the CV. Template matching finds events at 15 to 28 Hz in four recordings,
which suggests it counts noise at these thresholds. No setting has a basis in the paper, so the
claim is not judged.

## recording-properties

> Each tone was presented 18 times on average (10 to 63) per neuron; rest potential ranged from
> -58 to -92 mV (median -70) after junction potential correction, and input resistance from 19
> to 141 MOhm (median 64).

In [ ]:
props = []
for name, n in N.items():
    st = ac2.stable(n)
    sweeps = np.unique(n["sweep"][st])
    ok = ~ac2.spikes(n, "cell") & st
    trials_per_tone = pd.Series(n["tone"][ok]).value_counts()
    dv = np.median(n["prepulse_dv"][sweeps])
    props.append({"neuron": name, "reps": int(n["params"]["last_cycle"] - n["params"]["first_cycle"] + 1),
                  "mean trials per tone (no spikes)": trials_per_tone.reindex(ac2.TONE_ORDER).fillna(0).mean(),
                  "rest, sweeps (raw)": np.median(n["sweep_rest"][sweeps]),
                  "rest, 15-ms baselines (raw)": np.median(n["rbaseline"][st]),
                  "prepulse pA": n["info"]["prepulse_height"], "prepulse dV mV": dv,
                  "Rin MOhm": -dv / n["info"]["prepulse_height"] * 1000})
props = pd.DataFrame(props).set_index("neuron")
print(props.round(2).to_string())
r = props.reps
print(f"\nRepetitions: mean {r.mean():.2f} +- {r.std(ddof=1) / np.sqrt(len(r)):.2f} (SE), range {r.min()} to {r.max()}")
v = props["mean trials per tone (no spikes)"]
print(f"Variant, mean trials per tone without spike trials: mean {v.mean():.2f}, range {v.min():.1f} to {v.max():.1f}")
for jp in (12, 10):
    for col in ("rest, sweeps (raw)", "rest, 15-ms baselines (raw)"):
        x = props[col] - jp
        print(f"Rest potential, {col.split(',')[1].split('(')[0].strip()}, corrected by -{jp} mV: median {x.median():.1f},"
              f" range {x.max():.1f} to {x.min():.1f}; at or below -55 mV: {int((x <= -55).sum())} of {len(x)}")
x = props["rest, sweeps (raw)"] + 10
print(f"Documentation's direction (+10 mV, 'more depolarized'): median {x.median():.1f}, range {x.max():.1f} to {x.min():.1f}")
rin = props["Rin MOhm"]
print(f"Input resistance (last 20% of the prepulse against the {int(N[all_names[0]]['info']['prepulse_baselength'])} ms before it):"
      f" median {rin.median():.1f}, range {rin.min():.1f} to {rin.max():.1f} MOhm")
# Variant: the step response includes the voltage drop across the access resistance (no bridge
# balance, "no on-line series resistance compensation"); subtract the recorded access resistance.
rin2 = rin - cells.R_access.astype(float)
print(f"Variant, minus the recorded access resistance: median {rin2.median():.1f}, range {rin2.min():.1f} to {rin2.max():.1f} MOhm;"
      f" within 17.1-155.1: {int(rin2.between(17.1, 155.1).sum())} of {rin2.notna().sum()}")
print("Prepulse sizes by anesthetic:", props.groupby(cells.anesthetic)["prepulse pA"].unique().to_dict())

**Result: reproduced** (resting on the assumption that input resistance excludes the access
resistance). Repetitions per tone: 18.12 +- 1.93 (SE), 10 to 63, as in the paper. Rest
potential from the 15-ms baselines, corrected by -12 mV: median -70.0 mV, range -58.4 to -92.0,
as in the paper; from the sweeps outside the prepulse, median -68.6 and range -55.3 to -87.9
(within the widened range). Input resistance from the prepulse is 58 to 438 MOhm (median 138)
as measured; minus the recorded access resistance, which the uncompensated step includes, it is
19.1 to 140.6 MOhm (median 65.1), as in the paper (19 to 141, median 64). The step response
alone does not match.

## access-resistance-depth-no-effect

> Response variability is not correlated with access resistance (4.2 against 4.9 mV on either
> side of the 60 MOhm median) and does not vary systematically with recording depth.

In [ ]:
ra = cells.R_access.astype(float)
depth = cells.depth_um.astype(float)
med = ra.median()
print(f"Access resistance: median {med:.1f} MOhm, range {ra.min():.0f} to {ra.max():.0f}; neurons at the median: {list(ra.index[ra == med])}")
print(f"Depth (Sutter travel / sqrt 2): {depth.notna().sum()} of 33 recorded, {depth.min():.0f} to {depth.max():.0f} um (paper: 180 to 834)")
for label in ("main", "spikes -20 mV (6, claims file)"):
    q = S[label].q
    for placement, above in (("median cell with the upper group", ra >= med), ("median cell with the lower group", ra > med)):
        print(f"[{label}; {placement}] above median {mse(q[above])}; below {mse(q[~above])}")
    for var, x in (("access resistance", ra), ("depth", depth)):
        ok = x.notna()
        pr, pp = stats.pearsonr(x[ok], q[ok])
        sr, sp = stats.spearmanr(x[ok], q[ok])
        print(f"[{label}] {var}: Pearson r = {pr:.3f}, p = {pp:.3f}; Spearman rho = {sr:.3f}, p = {sp:.3f} (n = {ok.sum()})")

**Result: reproduced.** The median access resistance is 60 MOhm (10 to 320). <q> is 4.88 +- 0.71 mV
for the 17 neurons at or above the median and 4.27 +- 0.63 mV for the 16 below it, both within
the tolerances of 4.2 +- 0.7 and 4.9 +- 0.7 (the values fit the paper's numbers better the other
way round). Neither Pearson (r = 0.087, p = 0.631) nor Spearman (rho = 0.155, p = 0.390) finds a
correlation with access resistance, nor with depth over the 25 neurons with a recorded depth
(r = -0.273, p = 0.187; rho = -0.338, p = 0.098). With the -20 mV spike rule the Spearman
correlation with depth is significant (rho = -0.438, p = 0.028), so the depth part rests on the
spike-rule assumption.

## Summary

| Claim | Tested | Outcome |
|---|---|---|
| fig2-stereotyped-shape | yes | partially reproduced |
| fig3a-reliable-neuron | yes | reproduced |
| fig3b-total-variability-distribution | yes | reproduced (assumed) |
| total-variability-population | yes | reproduced (assumed) |
| lfp-culling-reduction | yes | reproduced (assumed) |
| fig5b-residual-lfp-psp-correlation | yes | reproduced |
| fig7a-lfp-culling-estimation | yes | partially reproduced (assumed) |
| psp-culling-reduction | yes | partially reproduced |
| fig4c-psp-culling-can-increase | yes | partially reproduced (assumed) |
| fig7b-private-best-estimate | yes | reproduced |
| culling-controls | yes | not reproduced |
| fig7d-large-responses | yes | reproduced |
| large-responses-controls | yes | not reproduced |
| cv-large-responses | yes | partially reproduced |
| lfp-cv-not-different | yes | reproduced |
| anesthesia-comparison | yes | partially reproduced |
| fig8-spontaneous-veto | yes | reproduced |
| overfitting-correction-example | yes | reproduced |
| private-shared-ranges | yes | partially reproduced |
| quantal-comparison | yes | reproduced |
| mepsp-size | yes, every reading run | blocked: methods underspecified |
| recording-properties | yes | reproduced (assumed) |
| access-resistance-depth-no-effect | yes | reproduced (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The paper links no code. Trials are each neuron's
stable epoch as given in the data set's `cell_specific_parameters.m` (its length matches the
paper's repetitions: mean 18.1, 10 to 63). Spike trials are found with the data set's
per-neuron threshold, an assumption: the claims file's -20 mV rule lowers <q> from 4.58 to 4.05
mV and is shown for every claim. The peak of the mean PSP is searched over the whole trace after
onset; the peak-to-trough height for culling is taken over the culling window; heights are not
re-measured after culling; when both cullings are applied each is decided on the full ensemble;
the overfitting correction averages 20 shuffles per neuron. Each open detail is run under the
readings the claims file lists, and the tables above show them. Input resistance is taken as the
prepulse response minus the recorded access resistance. The mEPSP detector is not given and the
result depends on it, so that claim is blocked.

**What was found about the data set.** It holds the 33 neurons, the 29 LFP flags, the anesthetic
groups and the six TTX recordings the paper describes, and with the per-neuron stable epochs and
spike thresholds of `cell_specific_parameters.m` the paper's population values follow closely
(<q> 4.58, 4.80 with LFP, 2.19 after PSP-based culling, 3.19 after LFP-based culling, 1.51 for
the best private estimate). Those files are not described for these neurons. The documentation
gives a junction potential of about 10 mV and says the true potentials are about 10 mV more
depolarized than recorded; the paper subtracts 12 mV, and only that reproduces its rest
potentials. It calls the prepulse height a voltage in mV, while the values are currents in pA.
Depth is missing for 8 of 33 neurons. The TTX recording 030401md02h, after its documented
division by 4, sits near +2.6 mV, which is not a membrane potential.